# Airflow Tutorial

This notebook walks through Apache Airflow core concepts by writing real DAG files that the running Airflow scheduler picks up automatically.

**Prerequisites:** start the stack with `docker compose up --build` from this directory.

- **Airflow UI** → http://localhost:8080 (admin / admin)
- **Jupyter**    → http://localhost:8888

---

## Contents
1. [Key concepts](#1-concepts)
2. [Connect via REST API](#2-connect)
3. [First DAG — PythonOperator & BashOperator](#3-first-dag)
4. [Task dependencies](#4-dependencies)
5. [TaskFlow API — the modern way](#5-taskflow)
6. [XCom — passing data between tasks](#6-xcom)
7. [Branching](#7-branching)
8. [Scheduling](#8-scheduling)
9. [Trigger & monitor runs via REST API](#9-rest)

## 0 — Imports & helpers

In [ ]:
import os, time, textwrap, json
import requests
from pathlib import Path
import pandas as pd

# --- Airflow REST API helpers ------------------------------------------------
BASE_URL  = os.environ.get("AIRFLOW_BASE_URL", "http://localhost:8080")
AUTH      = (
    os.environ.get("AIRFLOW_USER",     "admin"),
    os.environ.get("AIRFLOW_PASSWORD", "admin"),
)
HEADERS   = {"Content-Type": "application/json"}
DAGS_DIR  = Path("/tutorial/dags")   # mounted into Airflow at /opt/airflow/dags

def af(method, path, **kwargs):
    """Thin wrapper around the Airflow REST API."""
    resp = requests.request(method, f"{BASE_URL}/api/v1{path}", auth=AUTH, headers=HEADERS, **kwargs)
    resp.raise_for_status()
    return resp.json()

def write_dag(filename, source):
    """Write a DAG file and wait for Airflow to parse it."""
    (DAGS_DIR / filename).write_text(textwrap.dedent(source))
    print(f"Written: dags/{filename}")
    print("Waiting for scheduler to parse... (~5 s)")
    time.sleep(5)

print(f"Airflow base URL : {BASE_URL}")
print(f"DAGs directory   : {DAGS_DIR}")

## 1 — Key concepts <a id='1-concepts'></a>

| Concept | What it is |
|---|---|
| **DAG** | A Python file describing a workflow as a graph of tasks |
| **Task** | One unit of work in a DAG |
| **Operator** | A template that defines *what* a task does (Python, Bash, HTTP, SQL …) |
| **Scheduler** | Watches DAG files, decides when runs are due, queues tasks |
| **Executor** | Decides *how* tasks run — locally, on Celery workers, on Kubernetes … |
| **DAG Run** | One execution of a DAG (triggered manually or by schedule) |
| **Task Instance** | One execution of a single task inside a DAG Run |
| **XCom** | Small values tasks can push/pull to share data |

```
DAG file (Python)
  └── DAG object
        ├── Task A  (PythonOperator)
        ├── Task B  (BashOperator)
        └── Task C  (PythonOperator)
              A >> B >> C   ← dependency chain
```

## 2 — Connect via REST API <a id='2-connect'></a>

Airflow exposes a REST API at `/api/v1`. We will use it throughout to trigger runs and check status.
The UI at http://localhost:8080 shows the same data visually.

In [ ]:
health = af("GET", "/health")
print("Scheduler:", health["scheduler"]["status"])
print("Metadb   :", health["metadatabase"]["status"])

In [ ]:
dags = af("GET", "/dags")
print(f"DAGs currently loaded: {len(dags['dags'])}")
for d in dags["dags"]:
    print(f"  {d['dag_id']:40s}  paused={d['is_paused']}")

## 3 — First DAG — PythonOperator & BashOperator <a id='3-first-dag'></a>

The simplest possible DAG: two tasks that run in sequence.

- **PythonOperator** — calls any Python callable
- **BashOperator** — runs a shell command

DAG files are plain Python. Airflow's scheduler scans the `dags/` folder every few seconds and imports any file it finds.

In [ ]:
write_dag("01_hello_world.py", """
    from datetime import datetime
    from airflow.models import DAG
    from airflow.operators.python import PythonOperator
    from airflow.operators.bash import BashOperator

    def say_hello():
        print("Hello from PythonOperator!")
        return "hello"

    with DAG(
        dag_id="01_hello_world",
        start_date=datetime(2024, 1, 1),
        schedule=None,          # manual trigger only
        catchup=False,
        tags=["tutorial"],
    ) as dag:

        greet = PythonOperator(
            task_id="greet",
            python_callable=say_hello,
        )

        shout = BashOperator(
            task_id="shout",
            bash_command="echo 'Hello from BashOperator!'",
        )

        greet >> shout          # greet runs first, then shout
""")

In [ ]:
# Verify Airflow picked it up
dag = af("GET", "/dags/01_hello_world")
print(f"DAG id   : {dag['dag_id']}")
print(f"Is paused: {dag['is_paused']}")
print(f"Tags     : {dag['tags']}")

## 4 — Task dependencies <a id='4-dependencies'></a>

Dependencies define the execution order. Airflow only starts a task once all its upstream tasks have succeeded.

```
>>   set_downstream   A >> B   (A runs before B)
<<   set_upstream     B << A   (same thing)
```

You can express any DAG shape: sequential, parallel, fan-out, fan-in, diamond.

In [ ]:
write_dag("02_dependencies.py", """
    from datetime import datetime
    from airflow.models import DAG
    from airflow.operators.python import PythonOperator

    def task(name):
        def _run():
            print(f"Running: {name}")
        return _run

    with DAG(
        dag_id="02_dependencies",
        start_date=datetime(2024, 1, 1),
        schedule=None,
        catchup=False,
        tags=["tutorial"],
    ) as dag:

        start    = PythonOperator(task_id="start",    python_callable=task("start"))
        branch_a = PythonOperator(task_id="branch_a", python_callable=task("branch_a"))
        branch_b = PythonOperator(task_id="branch_b", python_callable=task("branch_b"))
        join     = PythonOperator(task_id="join",     python_callable=task("join"))

        # Diamond pattern: start fans out to a and b, join waits for both
        start >> [branch_a, branch_b] >> join
""")

## 5 — TaskFlow API — the modern way <a id='5-taskflow'></a>

The `@task` decorator (TaskFlow API, Airflow 2.0+) lets you write DAGs as plain Python functions — no explicit Operator instantiation, no manual XCom push/pull.

Returning a value from a `@task` function automatically pushes it to XCom. Passing that return value as an argument to another `@task` automatically pulls it.

In [ ]:
write_dag("03_taskflow.py", """
    from datetime import datetime
    from airflow.decorators import dag, task

    @dag(
        dag_id="03_taskflow",
        start_date=datetime(2024, 1, 1),
        schedule=None,
        catchup=False,
        tags=["tutorial"],
    )
    def pipeline():

        @task
        def extract() -> dict:
            # Simulate pulling data from somewhere
            data = {"values": [1, 2, 3, 4, 5], "source": "mock"}
            print(f"Extracted: {data}")
            return data          # automatically pushed to XCom

        @task
        def transform(raw: dict) -> dict:
            total  = sum(raw["values"])
            result = {"total": total, "mean": total / len(raw["values"])}
            print(f"Transformed: {result}")
            return result

        @task
        def load(result: dict):
            print(f"Loading result: {result}")
            # In real life: write to DB, GCS, etc.

        # Task dependencies are inferred from the data flow
        raw    = extract()
        result = transform(raw)
        load(result)

    pipeline()
""")

## 6 — XCom — passing data between tasks <a id='6-xcom'></a>

**XCom** (cross-communication) is Airflow's mechanism for tasks to share small values (strings, numbers, dicts). It is stored in the metadata database.

Two styles:
- **Classic:** explicit `xcom_push` / `xcom_pull` via the task context
- **TaskFlow:** returning / receiving values between `@task` functions (section 5)

> XCom is for small metadata — not large DataFrames or files. For large data, write to GCS/S3 and pass the path via XCom.

In [ ]:
write_dag("04_xcom.py", """
    from datetime import datetime
    from airflow.models import DAG
    from airflow.operators.python import PythonOperator

    def push_value(ti):
        # ti = TaskInstance — gives access to XCom, run_id, etc.
        ti.xcom_push(key="greeting", value="Hello from task A")

    def pull_value(ti):
        greeting = ti.xcom_pull(task_ids="push_task", key="greeting")
        print(f"Received via XCom: {greeting}")

    with DAG(
        dag_id="04_xcom",
        start_date=datetime(2024, 1, 1),
        schedule=None,
        catchup=False,
        tags=["tutorial"],
    ) as dag:

        push_task = PythonOperator(
            task_id="push_task",
            python_callable=push_value,
        )

        pull_task = PythonOperator(
            task_id="pull_task",
            python_callable=pull_value,
        )

        push_task >> pull_task
""")

## 7 — Branching <a id='7-branching'></a>

**BranchPythonOperator** lets a task decide at runtime which downstream task(s) to execute. It returns the `task_id` (or list of `task_id`s) to follow — all other branches are skipped.

Useful for: environment-specific paths, A/B logic, conditional loads.

In [ ]:
write_dag("05_branching.py", """
    from datetime import datetime
    from airflow.models import DAG
    from airflow.operators.python import PythonOperator, BranchPythonOperator
    from airflow.operators.empty import EmptyOperator

    def decide_branch():
        import random
        choice = random.choice(["path_a", "path_b"])
        print(f"Branching to: {choice}")
        return choice   # must match a task_id downstream

    with DAG(
        dag_id="05_branching",
        start_date=datetime(2024, 1, 1),
        schedule=None,
        catchup=False,
        tags=["tutorial"],
    ) as dag:

        start = EmptyOperator(task_id="start")

        branch = BranchPythonOperator(
            task_id="branch",
            python_callable=decide_branch,
        )

        path_a = PythonOperator(
            task_id="path_a",
            python_callable=lambda: print("Took path A"),
        )

        path_b = PythonOperator(
            task_id="path_b",
            python_callable=lambda: print("Took path B"),
        )

        # trigger_rule=none_failed_min_one_success lets the join run
        # even if one branch was skipped
        join = EmptyOperator(
            task_id="join",
            trigger_rule="none_failed_min_one_success",
        )

        start >> branch >> [path_a, path_b] >> join
""")

## 8 — Scheduling <a id='8-scheduling'></a>

The `schedule` parameter accepts:
- A **cron expression**: `'0 6 * * *'` (every day at 06:00)
- A **timedelta**: `timedelta(hours=1)`
- A **preset**: `'@daily'`, `'@hourly'`, `'@weekly'`, `'@monthly'`
- `None` — manual trigger only

**`catchup=False`** is important: without it Airflow will try to backfill all missed runs between `start_date` and now when you first deploy a DAG.

**`start_date`** is the earliest a run can be scheduled for. The first automatic run is triggered *after* `start_date + schedule interval`.

In [ ]:
write_dag("06_scheduling.py", """
    from datetime import datetime, timedelta
    from airflow.decorators import dag, task

    @dag(
        dag_id="06_scheduling",
        # Run every day at 07:00 UTC
        schedule="0 7 * * *",
        start_date=datetime(2024, 1, 1),
        catchup=False,          # do NOT backfill missed runs
        tags=["tutorial"],
        default_args={
            "retries": 2,
            "retry_delay": timedelta(minutes=5),
        },
    )
    def daily_job():

        @task
        def run():
            print("Daily job running!")

        run()

    daily_job()
""")
print("Open http://localhost:8080 and look at the 06_scheduling DAG.")
print("It is scheduled but will only auto-trigger tomorrow at 07:00 UTC.")
print("You can trigger it manually via the UI or the REST API cell below.")

## 9 — Trigger & monitor runs via REST API <a id='9-rest'></a>

Everything you can do in the UI can also be done via the REST API — useful for CI/CD pipelines, notebooks, or external orchestration.

In [ ]:
def trigger_dag(dag_id, conf=None):
    """Trigger a DAG run and return its run_id."""
    body = {"conf": conf or {}}
    run = af("POST", f"/dags/{dag_id}/dagRuns", json=body)
    print(f"Triggered {dag_id} -> run_id: {run['dag_run_id']}")
    return run["dag_run_id"]

def wait_for_run(dag_id, run_id, poll_seconds=3, timeout=120):
    """Poll until the DAG run reaches a terminal state."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        run = af("GET", f"/dags/{dag_id}/dagRuns/{run_id}")
        state = run["state"]
        print(f"  [{dag_id}] {run_id} -> {state}")
        if state in ("success", "failed", "upstream_failed"):
            return state
        time.sleep(poll_seconds)
    raise TimeoutError(f"Run {run_id} did not finish within {timeout}s")

def get_task_instances(dag_id, run_id):
    """Return a DataFrame of task instances for a run."""
    tis = af("GET", f"/dags/{dag_id}/dagRuns/{run_id}/taskInstances")
    rows = [
        {
            "task_id": ti["task_id"],
            "state":   ti["state"],
            "start":   ti["start_date"],
            "end":     ti["end_date"],
            "duration": ti["duration"],
        }
        for ti in tis["task_instances"]
    ]
    return pd.DataFrame(rows)

In [ ]:
# --- Trigger and watch each tutorial DAG ---
dag_ids = [
    "01_hello_world",
    "02_dependencies",
    "03_taskflow",
    "04_xcom",
    "05_branching",
]

results = {}
for dag_id in dag_ids:
    try:
        run_id = trigger_dag(dag_id)
        state  = wait_for_run(dag_id, run_id)
        results[dag_id] = {"run_id": run_id, "final_state": state}
    except Exception as e:
        results[dag_id] = {"run_id": None, "final_state": f"ERROR: {e}"}
    print()

pd.DataFrame(results).T

In [ ]:
# --- Inspect task instances for a specific run ---
dag_id = "03_taskflow"
run_id = results[dag_id]["run_id"]

if run_id:
    df = get_task_instances(dag_id, run_id)
    print(df.to_string(index=False))

In [ ]:
# --- Read XCom values pushed during a run ---
dag_id = "04_xcom"
run_id = results[dag_id]["run_id"]

if run_id:
    xcoms = af("GET", f"/dags/{dag_id}/dagRuns/{run_id}/taskInstances/push_task/xcomEntries")
    for x in xcoms.get("xcom_entries", []):
        detail = af("GET", f"/dags/{dag_id}/dagRuns/{run_id}/taskInstances/push_task/xcomEntries/{x['key']}")
        print(f"key={x['key']}  value={detail.get('value')}")

In [ ]:
# --- List all runs of a DAG ---
dag_id = "05_branching"
runs = af("GET", f"/dags/{dag_id}/dagRuns", params={"order_by": "-start_date", "limit": 10})
df_runs = pd.DataFrame([
    {"run_id": r["dag_run_id"], "state": r["state"], "start": r["start_date"]}
    for r in runs["dag_runs"]
])
print(df_runs.to_string(index=False))

---

## Summary

| Concept | Where defined | Key parameter |
|---|---|---|
| DAG | `with DAG(...)` or `@dag` | `dag_id`, `schedule`, `catchup` |
| PythonOperator | `PythonOperator(python_callable=fn)` | `task_id` |
| BashOperator | `BashOperator(bash_command='...')` | `task_id` |
| Dependencies | `>>` / `<<` / `[a, b]` | order of execution |
| TaskFlow | `@task` decorator | return value = XCom push |
| XCom | `ti.xcom_push/pull` or `@task` return | `key`, `task_ids` |
| Branching | `BranchPythonOperator` | returns downstream `task_id` |
| Scheduling | `schedule='0 7 * * *'` | cron, timedelta, or preset |
| REST API | `/api/v1/dags/{id}/dagRuns` | `POST` to trigger, `GET` to poll |